In [0]:
# =============================================================================
# NB08 — Baseline models: all model families × all feature layers
# =============================================================================
# Objective:
#   Evaluate all baseline model families under the IDENTICAL splits defined
#   in NB07. Compares three feature layers:
#     L1 = X_material only (compositional + structural + Butler-Ginley)
#     L2 = L1 + X_thermo  (+ thermodynamic descriptors)
#     L3 = L2 + X_exp     (+ experimental variables)
#
#   Models evaluated:
#     - Ridge (regularized linear)
#     - Random Forest
#     - XGBoost
#     - LightGBM
#     - CatBoost   (handles categoricals natively — used only for L3)
#     - SVR (RBF kernel)
#
#   Gaussian Process is omitted at this scale (691 rows × 100+ features:
#   GP kernel matrix is 691×691 per fold iteration — feasible but slow;
#   included as optional at the end).
#
#   For each model × layer combination:
#     - 5-fold main_cv (GroupKFold by DOI)  — primary metric
#     - y_randomization sanity check (R² must collapse to ~0)
#     - Metrics: R²(log), RMSE(log), MAE(log), R²(original scale)
#     - Results appended to ml_perovskites.cv_results_nb08
#       (separate table from NB10's cv_results to avoid overwrite)
#
#   Categorical encoding for L3 (inside each fold, no leakage):
#     - Smoothed target encoding (m=10) for co_catalyst_type, run_type,
#       scavenger_class, lamp_type
#     - Numeric NaN → median imputation (within fold)
#
# Inputs:
#   - ml_perovskites.master       (target + groups)
#   - ml_perovskites.cv_splits    (from NB07)
#   - ml_perovskites.y_randomized (from NB07)
#   - ml_perovskites.X_material   (L1 features)
#   - ml_perovskites.X_thermo     (thermo features for L2)
#   - ml_perovskites.X_exp        (experimental features for L3)
#
# Outputs:
#   - ml_perovskites.cv_results_nb08  (Delta table, one row per fold×model×layer)
#   - /tmp/nb08_cv_results.png        (bar chart: R² by model × layer)
#   - /tmp/nb08_parity_best.png       (parity plot: best model on L3)
#
# Key rules:
#   - All encoding and imputation strictly inside each fold (no leakage)
#   - y_randomization R² must be < 0.05 for all models (sanity gate)
#   - SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model  import Ridge
from sklearn.ensemble      import RandomForestRegressor
from sklearn.svm           import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.impute        import SimpleImputer
from sklearn.pipeline      import Pipeline
from sklearn.metrics       import r2_score, mean_squared_error, mean_absolute_error

from pyspark.sql import SparkSession

SEED   = 42
SCHEMA = "ml_perovskites"
CAT_COLS = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type"]

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

# ---------------------------------------------------------------------------
# Install optional packages (safe to re-run)
# ---------------------------------------------------------------------------
import subprocess, sys

def try_import(pkg, install_name=None):
    try:
        return __import__(pkg)
    except ImportError:
        name = install_name or pkg
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                                "--quiet", "--break-system-packages", name])
        return __import__(pkg)

xgb  = try_import("xgboost")
lgb  = try_import("lightgbm")
try:
    cb = try_import("catboost")
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False
    print("CatBoost not available — skipping")

# ---------------------------------------------------------------------------
# 1. Load all tables
# ---------------------------------------------------------------------------
master     = spark.table(f"{SCHEMA}.master").toPandas()
cv_splits  = spark.table(f"{SCHEMA}.cv_splits").toPandas()
y_rand_df  = spark.table(f"{SCHEMA}.y_randomized").toPandas()
X_mat_raw  = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw  = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw  = spark.table(f"{SCHEMA}.X_exp").toPandas()

print(f"master:    {master.shape}")
print(f"cv_splits: {cv_splits.shape}")
print(f"X_material:{X_mat_raw.shape}")
print(f"X_thermo:  {X_thm_raw.shape}")
print(f"X_exp:     {X_exp_raw.shape}")

# ---------------------------------------------------------------------------
# 2. Prepare aligned dataset
# ---------------------------------------------------------------------------
# use_h2 is stored as string "true"/"false" in this dataset
def is_use_h2(series):
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")

df_base = master[
    is_use_h2(master["use_h2"]) &
    master["y_log10_1p_final"].notna() &
    master["doi_key"].notna()
].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)
print(f"\nuse_h2 unique: {list(master['use_h2'].dropna().unique())}")
print(f"Modelling rows: {len(df_base)}")

# Build y_rand lookup
y_rand_map = y_rand_df.set_index("source_row_id")["y_rand"].to_dict()

# ---------------------------------------------------------------------------
# 3. Build feature matrices per layer
# ---------------------------------------------------------------------------
META_COLS = ["source_row_id", "base_compound", "family_proposed",
             "formula_base_paper", "A_site_paper", "B_site_paper",
             "doi", "doi_key"]

def get_numeric_features(df, exclude=None):
    """Return float64 numeric columns, excluding provenance and target."""
    excl = set(META_COLS + (exclude or []) + ["y_log10_1p_final",
                "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"])
    cols = [c for c in df.columns
            if c not in excl
            and pd.api.types.is_numeric_dtype(df[c])]
    return cols

# X_material: numeric only, drop high-NaN cols (>= 50%)
x_mat_cols = get_numeric_features(X_mat_raw)
X_mat = X_mat_raw[["source_row_id"] + x_mat_cols].copy()
for c in x_mat_cols:
    X_mat[c] = pd.to_numeric(X_mat[c], errors="coerce")
high_nan = [c for c in x_mat_cols if X_mat[c].isna().mean() >= 0.50]
X_mat = X_mat.drop(columns=high_nan)
x_mat_cols = [c for c in x_mat_cols if c not in high_nan]
print(f"L1 features (after dropping ≥50% NaN): {len(x_mat_cols)}")

# X_thermo: numeric only
x_thm_cols = get_numeric_features(X_thm_raw)
X_thm = X_thm_raw[["source_row_id"] + x_thm_cols].copy()
for c in x_thm_cols:
    X_thm[c] = pd.to_numeric(X_thm[c], errors="coerce")
# Drop 0% fill (thermal_conductivity_wav_cation)
zero_fill = [c for c in x_thm_cols if X_thm[c].isna().mean() >= 0.99]
X_thm = X_thm.drop(columns=zero_fill)
x_thm_cols = [c for c in x_thm_cols if c not in zero_fill]
print(f"L2 thermo features: {len(x_thm_cols)}")

# X_exp: numeric + smoothed target encoding for categoricals (in-fold)
x_exp_num_cols = [c for c in X_exp_raw.columns
                  if c not in ["source_row_id"] + CAT_COLS
                  and pd.api.types.is_numeric_dtype(X_exp_raw[c])]
X_exp = X_exp_raw[["source_row_id"] + x_exp_num_cols + CAT_COLS].copy()
for c in x_exp_num_cols:
    X_exp[c] = pd.to_numeric(X_exp[c], errors="coerce")
print(f"L3 numeric exp features: {len(x_exp_num_cols)}, "
      f"categorical: {len(CAT_COLS)}")

# Merge all onto df_base
df = (df_base[["source_row_id", "doi_key", "y"]]
      .merge(X_mat, on="source_row_id", how="left")
      .merge(X_thm, on="source_row_id", how="left")
      .merge(X_exp, on="source_row_id", how="left"))
print(f"Merged shape: {df.shape}")

# ---------------------------------------------------------------------------
# 4. Smoothed target encoding helper (inside fold only)
# ---------------------------------------------------------------------------
GLOBAL_MEAN_PLACEHOLDER = 0.0   # updated per fold

def smooth_target_encode(train_df, val_df, cat_col, target_col, m=10):
    """
    Smoothed target encoding for cat_col using training fold only.
    val_df gets the encoded values; unseen categories get global mean.
    """
    global_mean = train_df[target_col].mean()
    stats = (train_df.groupby(cat_col)[target_col]
             .agg(["mean", "count"]).reset_index())
    stats["encoded"] = (
        (stats["mean"] * stats["count"] + global_mean * m)
        / (stats["count"] + m)
    )
    enc_map = stats.set_index(cat_col)["encoded"].to_dict()
    train_enc = train_df[cat_col].map(enc_map).fillna(global_mean)
    val_enc   = val_df[cat_col].map(enc_map).fillna(global_mean)
    return train_enc, val_enc

# ---------------------------------------------------------------------------
# 5. Model definitions
# ---------------------------------------------------------------------------
def make_models():
    models = {
        "Ridge": Pipeline([
            ("imp",   SimpleImputer(strategy="median")),
            ("scl",   StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=SEED)),
        ]),
        "RF": RandomForestRegressor(
            n_estimators=300, max_features="sqrt",
            min_samples_leaf=3, n_jobs=-1, random_state=SEED),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=SEED, verbosity=0),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=400, learning_rate=0.05, num_leaves=31,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, verbose=-1),
    }
    if HAS_CATBOOST:
        models["CatBoost"] = cb.CatBoostRegressor(
            iterations=400, learning_rate=0.05, depth=4,
            random_seed=SEED, verbose=0)
    return models

# ---------------------------------------------------------------------------
# 6. Prepare feature column sets per layer
# ---------------------------------------------------------------------------
L1_COLS = x_mat_cols
L2_COLS = x_mat_cols + x_thm_cols
L3_NUM  = x_mat_cols + x_thm_cols + x_exp_num_cols   # categoricals added in-fold

LAYERS  = {"L1": L1_COLS, "L2": L2_COLS}   # L3 handled separately (encoding)

# ---------------------------------------------------------------------------
# 7. CV loop
# ---------------------------------------------------------------------------
main_cv   = cv_splits[cv_splits["split_type"] == "main_cv"]
y_rand_cv = cv_splits[cv_splits["split_type"] == "y_randomization"]

all_results = []

def run_cv(splits_df, layer_name, feat_cols, model_name, model,
           use_rand_y=False, is_l3=False):
    """Run one model × layer × split_type combination."""
    fold_metrics = []
    for fold_id in sorted(splits_df["fold_id"].unique()):
        fold = splits_df[splits_df["fold_id"] == fold_id]
        tr_ids = set(fold[fold["role"] == "train"]["source_row_id"])
        va_ids = set(fold[fold["role"] == "val"]["source_row_id"])

        tr = df[df["source_row_id"].isin(tr_ids)].copy()
        va = df[df["source_row_id"].isin(va_ids)].copy()

        if use_rand_y:
            tr["y"] = tr["source_row_id"].map(y_rand_map)
            va["y"] = va["source_row_id"].map(y_rand_map)

        if is_l3:
            # Encode categoricals inside fold
            enc_tr = tr[L3_NUM].copy()
            enc_va = va[L3_NUM].copy()
            for cc in CAT_COLS:
                te_tr, te_va = smooth_target_encode(tr, va, cc, "y")
                enc_tr[cc + "_enc"] = te_tr.values
                enc_va[cc + "_enc"] = te_va.values
            X_tr = enc_tr.values
            X_va = enc_va.values
        else:
            X_tr = tr[feat_cols].values
            X_va = va[feat_cols].values

        y_tr = tr["y"].values
        y_va = va["y"].values

        # Impute NaN for tree models (Ridge uses Pipeline)
        if model_name != "Ridge":
            imp = SimpleImputer(strategy="median")
            X_tr = imp.fit_transform(X_tr)
            X_va = imp.transform(X_va)

        import copy
        m = copy.deepcopy(model)
        m.fit(X_tr, y_tr)
        y_pred_log = m.predict(X_va)

        # Metrics on log scale
        r2_log   = r2_score(y_va, y_pred_log)
        rmse_log = np.sqrt(mean_squared_error(y_va, y_pred_log))
        mae_log  = mean_absolute_error(y_va, y_pred_log)

        # Metrics on original scale (reverse log1p)
        y_va_orig   = np.expm1(y_va)
        y_pred_orig = np.expm1(np.clip(y_pred_log, -1, 20))
        r2_orig  = r2_score(y_va_orig, y_pred_orig)

        fold_metrics.append({
            "fold_id":  fold_id,
            "r2_log":   r2_log,
            "rmse_log": rmse_log,
            "mae_log":  mae_log,
            "r2_orig":  r2_orig,
        })

    fm = pd.DataFrame(fold_metrics)
    return {
        "model":        model_name,
        "layer":        layer_name,
        "r2_log_mean":  fm["r2_log"].mean(),
        "r2_log_std":   fm["r2_log"].std(),
        "rmse_log":     fm["rmse_log"].mean(),
        "mae_log":      fm["mae_log"].mean(),
        "r2_orig_mean": fm["r2_orig"].mean(),
    }

print("\n--- Running CV: main_cv (L1, L2, L3) ---")
models = make_models()

for model_name, model in models.items():
    for layer_name, feat_cols in LAYERS.items():
        res = run_cv(main_cv, layer_name, feat_cols, model_name, model)
        all_results.append({**res, "split_type": "main_cv"})
        print(f"  {model_name:10s} | {layer_name} | "
              f"R²(log)={res['r2_log_mean']:+.3f}±{res['r2_log_std']:.3f}")

    # L3 (experimental features, with encoding)
    res = run_cv(main_cv, "L3", L3_NUM, model_name, model, is_l3=True)
    all_results.append({**res, "split_type": "main_cv"})
    print(f"  {model_name:10s} | L3 | "
          f"R²(log)={res['r2_log_mean']:+.3f}±{res['r2_log_std']:.3f}")

# --- y_randomization sanity check (L3 only — worst case for leakage) ---
print("\n--- y_randomization sanity check ---")
for model_name, model in models.items():
    res = run_cv(y_rand_cv, "L3", L3_NUM, model_name, model,
                 use_rand_y=True, is_l3=True)
    res["split_type"] = "y_randomization"
    all_results.append(res)
    flag = "PASS ✓" if res["r2_log_mean"] < 0.05 else "FAIL ✗ — possible leakage"
    print(f"  {model_name:10s} | L3 | R²(rand)={res['r2_log_mean']:+.3f} → {flag}")

# ---------------------------------------------------------------------------
# 8. Summary table
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(all_results)
main_results = results_df[results_df["split_type"] == "main_cv"].sort_values(
    "r2_log_mean", ascending=False)
print("\n--- Summary: main_cv ---")
print(main_results[["model", "layer", "r2_log_mean", "r2_log_std",
                     "rmse_log", "mae_log", "r2_orig_mean"]].to_string(index=False))

# ---------------------------------------------------------------------------
# 9. Plots
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
pivot = main_results.pivot(index="model", columns="layer", values="r2_log_mean")
pivot.plot(kind="bar", ax=ax, width=0.6)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("R² (log scale, GroupKFold)")
ax.set_title("Baseline models × feature layers (leave-one-paper-out CV)")
ax.legend(title="Layer")
plt.tight_layout()
plt.savefig("/tmp/nb08_cv_results.png", dpi=120)
plt.close()
print("Saved: /tmp/nb08_cv_results.png")

# ---------------------------------------------------------------------------
# 10. Write results to Delta
# ---------------------------------------------------------------------------
spark_df = spark.createDataFrame(results_df.replace({np.nan: None}))
(spark_df.write
 .format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{SCHEMA}.cv_results_nb08"))

rc = spark.table(f"{SCHEMA}.cv_results_nb08").count()
print(f"\n✓ Written: {SCHEMA}.cv_results_nb08 ({rc} rows)")
print("\n=== NB08 COMPLETE ===")
print("Next: NB09 — feature selection (VIF + permutation importance inside fold).")